# Give your agent the web with Tavily

One agent, built twice:

1. **Without web access.** It only knows what the model learned in training.
2. **With Tavily.** The same agent, plus two tools.

Stack: [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) for the agent, a model served by [Nebius Token Factory](https://tokenfactory.nebius.com/), and [Tavily](https://tavily.com) for the web.

## Setup

Copy `.env.example` to `.env` in the repo root and fill in `NEBIUS_API_KEY` and `TAVILY_API_KEY`.

In [2]:
import os
import time

from deepagents import create_deep_agent
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage
from langchain_nebius import ChatNebius

load_dotenv(find_dotenv())

missing = [k for k in ("NEBIUS_API_KEY", "TAVILY_API_KEY") if not os.getenv(k)]
assert not missing, (
    f"Missing {', '.join(missing)}. Copy .env.example to .env and add keys from "
    "https://tokenfactory.nebius.com/ and https://app.tavily.com/"
)

MODEL = "meta-models/muse-spark-open"

model = ChatNebius(
    model=MODEL,
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

SYSTEM_PROMPT = f"""Today is {time.strftime("%Y-%m-%d")}. You are a helpful assistant.
Answer the question directly. Cite a source URL for every fact you look up."""


def ask(agent, question):
    """Stream the agent run, print each tool call live, and display the answer."""
    answer = ""
    stream = agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates",
    )
    for update in stream:
        for partial in (update or {}).values():
            if not isinstance(partial, dict):
                continue
            for message in partial.get("messages") or []:
                if not isinstance(message, AIMessage):
                    continue
                for call in message.tool_calls or []:
                    print("→", call["name"], call["args"], flush=True)
                if not message.tool_calls and message.text.strip():
                    answer = message.text
    display(Markdown(answer))

## 1. An agent without web access

Pick a question whose answer changed after the model was trained.

In [3]:
QUESTION = "What are the three newest models on Nebius Token Factory, and when was each one added?"

agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT)

ask(agent, QUESTION)

→ ls {'path': '/'}
→ glob {'pattern': '**/*'}
→ grep {'pattern': 'Nebius', 'output_mode': 'files_with_matches'}


 As of my knowledge cutoff [2026-01-04], and without live browsing tools enabled in this environment (`ls` and file search returned no files and no `Nebius` data), I cannot retrieve the real-time catalog of Nebius Token Factory to verify the three newest models as of today, 2026-11-23.

Nebius Token Factory models are updated frequently, so the “newest” list changes week-to-week and requires live lookup. To get the accurate answer with added dates:

1. Go to the official Nebius Token Factory catalog:
   - https://nebius.com/platform/token-factory
   - https://docs.nebius.com/ — and search for “Token Factory models”

2. Sort/filter by **Date Added** or **Created** (newest first) — the catalog table typically shows Model Name, Provider/Author, and **Added On** date.

3. The top 3 rows after sorting are the three newest models and their added dates.

If you can paste a screenshot or copy the top rows of the catalog table here, I can help format them into a table with the names and added dates for you.

For verified, citable model history you would need to cite the specific Nebius Token Factory model page URLs, e.g. `https://nebius.com/.../token-factory/<model-name>` — each model’s detail page lists its `Added` / `Published` date which serves as the source citation per model.

## 2. The same agent with Tavily

The only change is the `tools` argument. `TavilySearch` finds sources (with recency and domain filters) and `TavilyExtract` reads full pages as markdown.

In [4]:
from langchain_tavily import TavilyExtract, TavilySearch

web_agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[TavilySearch(max_results=5, search_depth="advanced"), TavilyExtract()],
)

ask(web_agent, QUESTION)

→ tavily_search {'query': 'Nebius Token Factory newest models added', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Nebius AI Studio Token Factory models list', 'search_depth': 'advanced'}
→ tavily_extract {'urls': ['https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice', 'https://nebius.com/services/token-factory'], 'extract_depth': 'advanced'}
→ tavily_search {'query': 'site:nebius.com Token Factory new models release date 2025 2026', 'search_depth': 'advanced'}
→ tavily_search {'query': 'Reflection Beam Muse Spark Open Nemotron 3.5 Super VL Nebius Token Factory date announced', 'search_depth': 'advanced'}
→ tavily_search {'query': '"nebius.com/blog" Token Factory new models November 2025 December 2025 date', 'search_depth': 'advanced'}
→ tavily_extract {'urls': ['https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice'], 'extract_depth': 'advanced', 'query': 'date published added

Based on Nebius official announcements, the three newest models on Nebius Token Factory are the three US open-weight models added together in mid-October 2026:

### 1. Reflection Beam
*   **Added:** Week of October 16, 2026 - announced as `available on Nebius Token Factory from day zero` [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice](https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice)
*   **Announcement date:** `Posted: 2026-10-16T14:49:02+00:00` - "Reflection Beam is our third model with Day 0 support, together with NVIDIA Nemotron 3.5 Super VL and Meta Muse Spark Open, on Nebius Token Factory within 24 hours" [https://x.com/hanrui_w/status/2111107154619101681](https://x.com/hanrui_w/status/2111107154619101681)

### 2. Muse Spark Open from Meta
*   **Added:** Week of October 16, 2026 - listed as `This week, Reflection Beam, Muse Spark Open from Meta, and NVIDIA Nemotron 3.5 Super VL join ... with all three available on Nebius Token Factory from day zero` [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice](https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice)
*   **Announcement date:** `Posted: 2026-10-16T16:20:06+00:00` - "Three new models from US labs are available on Nebius Token Factory @nebiustf from day zero" includes `Meta’s Muse Spark Open` [https://x.com/hanrui_w/status/2111130072560349527](https://x.com/hanrui_w/status/2111130072560349527)

### 3. NVIDIA Nemotron 3.5 Super VL
*   **Added:** Week of October 16, 2026 - same `Three new open-weight models from US companies ...` batch [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice](https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice)
*   **Announcement date:** `Posted: 2026-10-16T16:20:06+00:00` - listed in same X post as `NVIDIA Nemotron 3.5 Super VL — Brings visual understanding to document and web workflows` [https://x.com/hanrui_w/status/2111130072560349527](https://x.com/hanrui_w/status/2111130072560349527)

Additional confirmation that the batch was described as:

> `Three new US open-weight models. Day 0 availability. More choice for builders on Nebius Token Factory` [https://www.linkedin.com/posts/nebius_three-new-us-open-weight-models-day-0-availability-activity-7516865298566660098-U5Xb](https://www.linkedin.com/posts/nebius_three-new-us-open-weight-models-day-0-availability-activity-7516865298566660098-U5Xb)

And summarized on Nebius Blog as:

> `The open model ecosystem is growing in both capability and breadth. This week, Reflection Beam, Muse Spark Open from Meta, and NVIDIA Nemotron 3.5 Super VL join that momentum, with all three available on Nebius Token Factory from day zero` [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice](https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice)


## 3. A use case: competitor brief

The same agent handles a multi-step task. It searches with recency filters, extracts official pages in full, and writes a cited brief. Swap the company name and run it again.

In [5]:
ask(
    web_agent,
    """Write a one-page brief on Palo Alto Networks.
Cover: what it does, pricing, the two closest competitors, and news from the last 30 days.""",
)

→ tavily_search {'query': 'Palo Alto Networks what does company do overview products', 'search_depth': 'basic'}
→ tavily_search {'query': 'Palo Alto Networks pricing plans 2026', 'search_depth': 'basic'}
→ tavily_search {'query': 'Palo Alto Networks closest competitors', 'search_depth': 'basic'}
→ tavily_search {'query': 'Palo Alto Networks news last 30 days October November 2026', 'search_depth': 'advanced', 'time_range': 'month', 'topic': 'news'}


**Palo Alto Networks (NASDAQ: PANW) — One-Page Brief — November 23, 2026**

### 1. What It Does

Palo Alto Networks, Inc. is an American multinational cybersecurity company with headquarters in Santa Clara, California [https://en.wikipedia.org/wiki/Palo_Alto_Networks].

Its core product is a platform that includes advanced firewalls and cloud-based offerings that extend those firewalls to cover other aspects of security [https://en.wikipedia.org/wiki/Palo_Alto_Networks].

The company serves over 70,000 organizations in over 150 countries, including 85 of the Fortune 100 [https://en.wikipedia.org/wiki/Palo_Alto_Networks]. It is home to the Unit 42 threat research team [https://en.wikipedia.org/wiki/Palo_Alto_Networks] and in June 2018 former Google and SoftBank executive Nikesh Arora joined as Chairman and CEO [https://en.wikipedia.org/wiki/Palo_Alto_Networks].

Palo Alto describes itself as the global AI cybersecurity leader [https://investors.paloaltonetworks.com/news-releases/news-release-details/palo-alto-networks-announce-fiscal-first-quarter-2027-financial].

Business areas include Cybersecurity Software, Network Security, Cloud Security, Identity and Access Management, and Security Operations and Incident Response [https://tavily.com/knowledge/company/iB6hx4PLn9Iy].

The company provides cybersecurity services to enterprises, government and service providers including zero trust network security, cloud security, security analytics and automation, threat intelligence, security consulting, and secure access service edge (SASE) [https://www.globaldata.com/company-profile/palo-alto-networks-inc].

Portfolio brands are **Strata** (network security), **Prisma** (SASE / cloud), and **Cortex** (SOC) [https://www.globaldata.com/company-profile/palo-alto-networks-inc].

Key products and services include:
* Next-Generation Firewalls — Hardware Firewalls, Software Firewalls, PAN-OS, Panorama [https://www.paloaltonetworks.com/about-us]
* Cloud Delivered Security Services — Advanced Threat Prevention, Advanced URL Filtering, Advanced WildFire, Advanced DNS Security, Enterprise Data Loss Prevention [https://www.paloaltonetworks.com/about-us]
* Secure Access Service Edge — Prisma SASE, Prisma Access, Prisma SD-WAN, Prisma Browser [https://www.paloaltonetworks.com/about-us]
* Cloud Security — Cortex Cloud (formerly Prisma Cloud), Cloud Posture and Runtime Security [https://tavily.com/knowledge/company/iB6hx4PLn9Iy]
* Extended Detection and Response, Security Orchestration Automation and Response — Cortex XSIAM, Cortex XDR, Cortex XSOAR, Cortex Xpanse [https://tavily.com/knowledge/company/iB6hx4PLn9Iy]
* Next-Generation Identity Security and Threat Intel/Incident Response via Unit 42 [https://tavily.com/knowledge/company/iB6hx4PLn9Iy]

The company states it powers platforms by Precision AI [https://origin-www.paloaltonetworks.ca].

### 2. Pricing

Palo Alto sells most products as subscriptions or term licenses [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products]. Pricing varies with organization size, feature requirements and contract terms, so a custom quote from Palo Alto or an authorized reseller is required [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products].

Published October 2026 marketplace pricing:

| Product | Published Price, October 2026 | Licensing |
|---|---|---|
| PA-Series hardware firewalls | Quote only | Appliance plus subscriptions and support [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] |
| VM-Series virtual firewall | From $1.35 per hour on AWS Marketplace [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] | Hourly, or Software NGFW credits [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] |
| Cloud NGFW for AWS | $1.50 per hour plus $0.065 per GB [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] | Hourly plus traffic, or credits [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] |
| Prisma Access | Quote only [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] | Per mobile user or per Mbps [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] |
| Cortex Cloud, Cortex XDR, Cortex XSOAR | Quote only [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] | Quote [https://underdefense.com/industry-pricings/palo-alto-networks-pricing-ultimate-guide-for-security-products] |

Enterprise benchmark data for 2026 negotiated deals:

* NGFW PA-3000 series: $25K to $80K/unit list [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing]. Typical discount 30 to 40% off list, 38 to 48% off with Platformization [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing].
* Annual NGFW software subscriptions: 30 to 50% of hardware per year [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing].
* Prisma Access (per user/year): $60 to $150 list, $35 to $90 average enterprise, $28 to $70 Platformization rate [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing].
* Cortex XDR (per endpoint/year): $35 to $55 list, $25 to $40 average enterprise, $18 to $30 Platformization rate [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing].
* Palo Alto's Platformization offer delivers 35 to 55% aggregate savings versus buying each product separately at standard enterprise pricing [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing], with very large enterprises spending $10M to $50M+ annually getting fully custom pricing [https://vendorbenchmark.com/vendors/palo-alto-networks-pricing].

Third-party averages: SMB plans average around $15,264 per year and Enterprise plans average around $1,434,738 per year based on annual average spend [https://www.spendhound.com/marketplace/palo-alto-networks-pricing].

Comparison vs peers (mid-range):
* NGFW appliance: Palo Alto $20,000–$50,000 list vs Fortinet $10,000–$30,000 and Cisco $15,000–$40,000 [https://www.vendr.com/marketplace/palo-alto-networks]
* Annual Advanced Threat Prevention: Palo Alto $8,000–$20,000 per firewall vs Fortinet $4,000–$12,000 [https://www.vendr.com/marketplace/palo-alto-networks]
* SASE per user/year: Palo Alto $50–$150 vs Cisco $40–$120 [https://www.vendr.com/marketplace/palo-alto-networks]
* Estimated 3-year total (5 firewalls, mid-tier): Palo Alto $250,000–$450,000 vs Fortinet $150,000–$300,000 [https://www.vendr.com/marketplace/palo-alto-networks]
* Estimated 3-year total SASE (500 users): Palo Alto $200,000–$400,000 vs Cisco $180,000–$350,000 [https://www.vendr.com/marketplace/palo-alto-networks]
* Endpoint (2,000 endpoints, 3-yr): Palo Alto Cortex XDR $300,000–$550,000 ($40–$100 per endpoint/year) vs CrowdStrike Falcon $350,000–$650,000 ($50–$120 per endpoint/year) [https://www.vendr.com/marketplace/palo-alto-networks]

### 3. Two Closest Competitors

Palo Alto operates in a highly competitive cybersecurity landscape [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026].

Key peers include Fortinet, Inc. (FTNT), Zscaler, Inc. (ZS), CrowdStrike Holdings, Inc. (CRWD), SentinelOne, Cloudflare, Okta and Akamai [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026].

Market cap comparison: PANW $134.68B, FTNT $61.91B, ZS $26.38B, CRWD $108.80B [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026].

**1. Fortinet (FTNT) – Network Security / NGFW Head-to-Head**
Fortinet is the closest direct infrastructure rival in Software - Infrastructure [https://www.hudson-labs.com/research/palo-alto-networks-competitors-panw-peers-2026]. Both compete for next-gen firewall appliances and threat-prevention subscriptions. Fortinet typically undercuts on list price with hardware $10K-$30K vs Palo Alto $20K-$50K [https://www.vendr.com/marketplace/palo-alto-networks], positioning on lower total cost of ownership.

**2. CrowdStrike (CRWD) – Endpoint & SOC Platform; Zscaler (ZS) – SASE**
CrowdStrike is the primary endpoint/XDR rival; CrowdStrike is cited among Palo Alto's main competitors alongside Fortinet [https://www.marketbeat.com/stocks/NASDAQ/PANW/competitors-and-alternatives]. Zscaler is the pure-play SASE challenger to Prisma Access. Palo Alto vs CrowdStrike endpoint pricing is near parity but feature bundling differs: XDR/EDR capabilities included in Pro tier for Cortex vs included in higher tiers for Falcon [https://www.vendr.com/marketplace/palo-alto-networks].

### 4. News – Last 30 Days (Oct 23 – Nov 23, 2026)

**Acquisition of Applied Compute – Nov. 18, 2026:** Palo Alto Networks announced it entered into a definitive agreement to acquire Applied Compute [https://www.prnewswire.com/news-releases/palo-alto-networks-to-acquire-applied-compute-bringing-secure-open-weight-ai-to-the-enterprise-302944421.html]. Applied Compute is a leading platform for post-training and inference of open-weight AI models [https://www.prnewswire.com/news-releases/palo-alto-networks-to-acquire-applied-compute-bringing-secure-open-weight-ai-to-the-enterprise-302944421.html]. The combination will help enterprises train, adapt, and deploy open-source and open-weight AI securely, across public and sovereign clouds [https://www.prnewswire.com/news-releases/palo-alto-networks-to-acquire-applied-compute-bringing-secure-open-weight-ai-to-the-enterprise-302944421.html].

**Fiscal Q1 2027 Earnings – Nov. 18, 2026:** Palo Alto announced it will release financial results for its fiscal first quarter ended October on Nov. 18, 2026 [https://investors.paloaltonetworks.com/news-releases/news-release-details/palo-alto-networks-announce-fiscal-first-quarter-2027-financial], and reported fiscal first quarter 2027 financial results on Nov. 18 [https://www.prnewswire.com/news-releases/palo-alto-networks-to-acquire-applied-compute-bringing-secure-open-weight-ai-to-the-enterprise-302944421.html]. Preview coverage noted Palo Alto beat revenue expectations last quarter reporting $3.41 billion, up 34.4% year on year [https://stockstory.org/us/stocks/nasdaq/panw/news/earnings/palo-alto-networks-reports-q3-everything-you-need-to-know-ahead-of-earnings] and the market expected revenue to grow 33.6% year on year [https://stockstory.org/us/stocks/nasdaq/panw/news/earnings/palo-alto-networks-reports-q3-everything-you-need-to-know-ahead-of-earnings]. Full-year 2026 context: revenue rose 24% to $11.48 billion, NGS ARR reached $9.1 billion, RPO climbed 34% to $21.2 billion, with fiscal 2027 guidance of $14.1–$14.2 billion revenue [https://tickeron.com/blogs/palo-alto-networks-panw-can-the-stock-deliver-a-34-move-to-the-420-analyst-target-32209].

Stock closed at $343.45 on Nov. 20, 2026 [https://www.marketbeat.com/stocks/NASDAQ/PANW], with the company scheduled for the Barclays 24th Annual Global Technology Conference Dec. 8, 2026 [https://www.marketbeat.com/stocks/NASDAQ/PANW].

